# SFT → verified pairs → DPO: Colab quick run

Shortened pipeline on a free T4 (Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all). Expect ~2h in total.

In [ ]:
%cd /content
import os
if not os.path.exists('/content/sft-dpo-math'):  # safe to re-run
    !git clone https://github.com/aymenblk04/sft-dpo-math.git
%cd /content/sft-dpo-math
!git pull -q
!pip install -q -r requirements.txt && pip install -q -e .

`run()` streams a command's output and **stops the notebook if it fails**, so an error shows up where it happens.

In [ ]:
import subprocess

def run(cmd):
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'command failed (exit {p.returncode}): {cmd}')

## 1. SFT (1 epoch on 2000 questions)

In [ ]:
run('python scripts/train_sft.py --config configs/sft.yaml train.bf16=false train.fp16=true data.train_samples=2000 train.num_train_epochs=1')

## 2. Sample 4 answers for 500 questions and build verified pairs

In [ ]:
run('python scripts/build_pairs.py --config configs/sampling.yaml num_questions=500 num_samples=4')
print(open('data/generated/pairs.stats.json').read())

## 3. DPO

In [ ]:
run('python scripts/train_dpo.py --config configs/dpo.yaml train.bf16=false train.fp16=true')

## 4. Evaluate base, SFT and DPO on 300 test questions

In [ ]:
for name, path in [('base', 'Qwen/Qwen2.5-0.5B-Instruct'), ('sft', 'outputs/sft/merged'), ('dpo', 'outputs/dpo/merged')]:
    run(f'python scripts/evaluate.py --model {path} --num-samples 300 --out results/{name}')

In [ ]:
run('python scripts/plot.py --runs dpo')
from IPython.display import Image, Markdown, display
display(Markdown(open('figures/results.md').read()))
display(Image('figures/dpo_curves.png'))

In [ ]:
run('zip -qr results.zip results figures data/generated/pairs.stats.json outputs/sft/trainer_state.json outputs/dpo/trainer_state.json')
from google.colab import files
files.download('results.zip')